# Benchmark Evaluation — Tiny LLM

**Tiny LLM Challenge · Evaluation Suite**

This notebook runs perplexity and benchmark evaluations on a trained checkpoint.
Benchmarks: Wikitext-103 PPL, OpenWebText PPL, HellaSwag, WinoGrande, OpenBookQA, LAMBADA.

---
## 0 · Setup & Configuration

In [ ]:
import sys
import logging
from pathlib import Path

import torch
import numpy as np

# Ensure the project root is on the path
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "pyproject.toml").exists() and (p / "src").exists()),
    _cwd,
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from tiny_llm.utils.device import resolve_device
from tiny_llm.utils.seed import set_seed

# Configure logging for notebook output
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(name)s | %(message)s")
logger = logging.getLogger("benchmark_eval")

device = resolve_device()
set_seed(42)
print(f"Device: {device}")

---
## 1 · Load Model from Checkpoint

In [ ]:
from tiny_llm.model.transformer import GPTModel
from tiny_llm.training.checkpoint import load_checkpoint

# ── Set your checkpoint path here ──
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "small" / "step_000100.pt"
# Fallback config file (for older checkpoints that lack model_config)
CONFIG_PATH = PROJECT_ROOT / "configs" / "small.toml"

# Peek at metadata to get model config
meta = load_checkpoint(CHECKPOINT_PATH, model=None, device=device)
model_config = meta["model_config"]

if model_config is None:
    # Older checkpoint — reconstruct model config from the config file
    from tiny_llm.utils.config import load_config
    print(f"[info] Checkpoint lacks model_config, loading from {CONFIG_PATH.name}")
    cfg = load_config(str(CONFIG_PATH))
    model_config = cfg.model.model_dump()

print(f"Checkpoint: {CHECKPOINT_PATH.name}")
print(f"  Step: {meta['step']:,}")
print(f"  Val loss: {meta['val_loss']}")
print(f"  Tokens seen: {meta['tokens_seen']:,}")
print(f"  Model config: {model_config}")

# Build model and load weights
model = GPTModel(model_config)
load_checkpoint(CHECKPOINT_PATH, model, device=device)
model.to(device)
model.eval()

# Parameter count
n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nParameters: {n_params:,} total, {n_trainable:,} trainable")

---
## 2 · Perplexity — OpenWebText Test Split

In [ ]:
from tiny_llm.eval.perplexity import evaluate_perplexity

OWT_TEST_PATH = PROJECT_ROOT / "data" / "tokenized" / "test.bin"

ppl_owt = evaluate_perplexity(
    model,
    data_path=str(OWT_TEST_PATH),
    device=device,
    context_length=model_config["context_length"],
)
print(f"\nOpenWebText test PPL: {ppl_owt:.2f}")

---
## 3 · Perplexity — Wikitext-103

In [ ]:
ppl_wiki = evaluate_perplexity(
    model,
    data_path="wikitext-103",
    device=device,
    context_length=model_config["context_length"],
)
print(f"\nWikitext-103 test PPL: {ppl_wiki:.2f}")

---
## 4 · HellaSwag

In [ ]:
from tiny_llm.eval.benchmarks import evaluate_hellaswag

results_hs = evaluate_hellaswag(model, device)
print(f"\nHellaSwag results:")
print(f"  Log-likelihood accuracy: {results_hs['accuracy_ll']:.4f}")
print(f"  Generation accuracy:     {results_hs['accuracy_gen']:.4f}")
print(f"  N examples:              {results_hs['n_examples']:,}")

---
## 5 · WinoGrande

In [ ]:
from tiny_llm.eval.benchmarks import evaluate_winogrande

results_wg = evaluate_winogrande(model, device)
print(f"\nWinoGrande results:")
print(f"  Log-likelihood accuracy: {results_wg['accuracy_ll']:.4f}")
print(f"  Generation accuracy:     {results_wg['accuracy_gen']:.4f}")
print(f"  N examples:              {results_wg['n_examples']:,}")

---
## 6 · OpenBookQA

In [ ]:
from tiny_llm.eval.benchmarks import evaluate_openbookqa

results_obqa = evaluate_openbookqa(model, device)
print(f"\nOpenBookQA results:")
print(f"  Log-likelihood accuracy: {results_obqa['accuracy_ll']:.4f}")
print(f"  Generation accuracy:     {results_obqa['accuracy_gen']:.4f}")
print(f"  N examples:              {results_obqa['n_examples']:,}")

---
## 7 · LAMBADA

In [ ]:
from tiny_llm.eval.benchmarks import evaluate_lambada

results_lam = evaluate_lambada(model, device)
print(f"\nLAMBADA results:")
print(f"  Log-likelihood accuracy: {results_lam['accuracy_ll']:.4f}")
print(f"  Generation accuracy:     {results_lam['accuracy_gen']:.4f}")
print(f"  N examples:              {results_lam['n_examples']:,}")

---
## 8 · Results Summary

In [ ]:
import pandas as pd

# Perplexity results
ppl_rows = [
    {"Benchmark": "OpenWebText (test)", "Metric": "PPL", "Value": f"{ppl_owt:.2f}"},
    {"Benchmark": "Wikitext-103 (test)", "Metric": "PPL", "Value": f"{ppl_wiki:.2f}"},
]

# Benchmark results
bench_data = [
    ("HellaSwag", results_hs),
    ("WinoGrande", results_wg),
    ("OpenBookQA", results_obqa),
    ("LAMBADA", results_lam),
]

bench_rows = []
for name, res in bench_data:
    bench_rows.append({
        "Benchmark": name,
        "LL Accuracy": f"{res['accuracy_ll']:.4f}",
        "Gen Accuracy": f"{res['accuracy_gen']:.4f}",
        "N Examples": f"{res['n_examples']:,}",
    })

# Random baselines for reference
random_baselines = {
    "HellaSwag": 0.25,
    "WinoGrande": 0.50,
    "OpenBookQA": 0.25,
    "LAMBADA": 0.0,
}
for row in bench_rows:
    row["Random Baseline"] = f"{random_baselines[row['Benchmark']]:.2f}"

print("=" * 60)
print("PERPLEXITY RESULTS")
print("=" * 60)
df_ppl = pd.DataFrame(ppl_rows)
print(df_ppl.to_string(index=False))

print(f"\n{'=' * 60}")
print("BENCHMARK RESULTS")
print("=" * 60)
df_bench = pd.DataFrame(bench_rows)
print(df_bench.to_string(index=False))

print(f"\nCheckpoint: {CHECKPOINT_PATH.name} (step {meta['step']:,})")

---
## 9 · Save Results to `eval_outputs/`

Persists all scores as a timestamped JSON file for cross-run comparison.

In [ ]:
from tiny_llm.eval.results import save_eval_results

output_file = save_eval_results(
    checkpoint_path=CHECKPOINT_PATH,
    meta=meta,
    model_config=model_config,
    n_params=n_params,
    perplexity={"openwebtext_test": ppl_owt, "wikitext_103": ppl_wiki},
    benchmarks={
        "hellaswag": results_hs,
        "winogrande": results_wg,
        "openbookqa": results_obqa,
        "lambada": results_lam,
    },
    output_dir=PROJECT_ROOT / "eval_outputs",
)

print(f"Results saved to: {output_file.relative_to(PROJECT_ROOT)}")

# Show saved content
import json
with open(output_file) as f:
    saved = json.load(f)
print(f"\n  Model: {saved['model_name']} | Step: {saved['step']:,} | Params: {saved['n_params']:,}")
print(f"\n  Perplexity:")
for k, v in saved["perplexity"].items():
    print(f"    {k}: {v:.2f}")
print(f"\n  Benchmarks (LL accuracy):")
for k, v in saved["benchmarks"].items():
    print(f"    {k}: {v['accuracy_ll']:.4f}")

---
## 10 · Multi-Checkpoint Comparison (Optional)

Uncomment and modify the cell below to compare multiple checkpoints.

In [ ]:
# # ── Compare multiple checkpoints ──
# checkpoint_paths = [
#     PROJECT_ROOT / "checkpoints" / "step_001000.pt",
#     PROJECT_ROOT / "checkpoints" / "step_003000.pt",
#     PROJECT_ROOT / "checkpoints" / "best.pt",
# ]
#
# comparison_rows = []
# for ckpt_path in checkpoint_paths:
#     if not ckpt_path.exists():
#         print(f"Skipping {ckpt_path.name} (not found)")
#         continue
#
#     meta_cmp = load_checkpoint(ckpt_path, model=None, device=device)
#     model_cmp = GPTModel(meta_cmp["model_config"])
#     load_checkpoint(ckpt_path, model_cmp, device=device)
#     model_cmp.to(device)
#     model_cmp.eval()
#
#     ppl = evaluate_perplexity(
#         model_cmp, str(OWT_TEST_PATH), device,
#         context_length=meta_cmp["model_config"]["context_length"],
#     )
#     comparison_rows.append({
#         "Checkpoint": ckpt_path.name,
#         "Step": f"{meta_cmp['step']:,}",
#         "Val Loss": f"{meta_cmp['val_loss']:.4f}" if meta_cmp['val_loss'] else "N/A",
#         "OWT Test PPL": f"{ppl:.2f}",
#     })
#     del model_cmp  # free memory
#
# if comparison_rows:
#     df_cmp = pd.DataFrame(comparison_rows)
#     print(df_cmp.to_string(index=False))